# Analisi dei dati

Analisi comparativa delle strategie di esplorazione del maze, secondo le metriche di valutazione definite nella **sezione 4 della proposta** (*Risultati attesi e metriche di valutazione*, `docs/Rob_26_proposal.md`).

L'obiettivo è verificare le ipotesi formulate in fase di proposta:

- **A\*** dovrebbe guidare l'agente verso percorsi di costo inferiore, ma con frequenti ri-pianificazioni alla scoperta di nuovi muri, risultando potenzialmente meno efficiente in numero di mosse;
- **Flood fill / D\*-Lite** dovrebbe navigare in modo più diretto verso il goal, con minori revisioni del piano;
- le prestazioni dovrebbero degradare in modo contenuto (robustezza) al crescere della complessità dello scenario, misurata tramite il **detour index** dei goal posizionati (`src/goal_placement.py`).

## Dataset

I dati provengono dai log JSON prodotti da `MetricsLogger` (`src/metrics/logger.py`) e salvati in `results/logs/`, uno per ogni run *(algoritmo, maze, scenario)*. Ogni file contiene:

- **metriche scalari**: `total_moves`, `forward_moves`, `turns`, `distinct_cells_visited`, `total_visits`, `execution_time_s`;
- **replanning**: `replanning_events`, `cumulative_planning_time_s`, `cumulative_nodes_expanded`;
- **matrici** `wall_matrix` e `visit_matrix` (mappa interna costruita dall'agente e conteggio visite per cella);
- **scenario**: file del maze, numero di goal `k` e coppie *(cella, detour index)* dei goal posizionati.

I maze considerati (in `mazes/txt/`) sono:

- `museum.txt`
- `2017apec.txt`
- `2015japan.txt`
- '2013japan.txt'
- 93japx.txt
- 93apec.txt

Per ciascun maze le run coprono un numero crescente di goal (`k = 1, 2, ...`), posizionati automaticamente massimizzando il detour index — così la difficoltà dello scenario è controllata e quantificata.

## 1. Setup e caricamento dei dati

Carichiamo tutti i log JSON da `results/logs/` in un unico DataFrame, con una riga per run e le metriche scalari come colonne. Dal campo `scenario` estraiamo il maze, `k` e il detour index massimo/medio dei goal, che useremo come asse di difficoltà. Le matrici (`wall_matrix`, `visit_matrix`) le teniamo da parte in una struttura separata per le heatmap.

*Cosa faremo qui:* import, glob dei file, costruzione del DataFrame, controllo rapido di completezza (tutte le combinazioni algoritmo × maze × k presenti?).

## 2. Efficienza ed efficacia di esplorazione

**Efficienza**: numero totale di mosse (passi + rotazioni) necessarie per raggiungere il goal. **Efficacia**: numero di celle distinte visitate al raggiungimento del goal, espresso anche come percentuale della dimensione del maze.

*Cosa faremo qui:* tabella riassuntiva per algoritmo × maze e barplot comparativi di `total_moves` e `distinct_cells_visited` (%). Prima verifica dell'ipotesi: A* usa più mosse di D*-Lite/flood fill?

## 3. Ridondanza di esplorazione

Confrontiamo `total_visits` (ogni accesso conta, anche le riviste) con `distinct_cells_visited`: il rapporto tra i due misura quanto l'agente ripercorre celle già note. Un algoritmo che ri-pianifica spesso dovrebbe mostrare ridondanza più alta.

*Cosa faremo qui:* barplot comparativo delle visite totali per strategia (come previsto dalla proposta) e indice di ridondanza `total_visits / distinct_cells_visited` per algoritmo × maze.

## 4. Qualità della mappa interna

Dalla `wall_matrix` esportata a fine run calcoliamo **offline** (BFS) il percorso minimo sulla mappa parziale costruita dall'agente e lo confrontiamo con il percorso minimo sul maze completo. Il rapporto tra le due lunghezze misura la qualità della conoscenza acquisita: un rapporto vicino a 1 indica che l'agente ha scoperto abbastanza del maze da conoscere un percorso (quasi) ottimo.

*Cosa faremo qui:* calcolo dei due percorsi minimi per ogni run, tabella dei rapporti per algoritmo × maze × k.

## 5. Heatmap di esplorazione

Visualizzazione qualitativa della `visit_matrix`: per ogni maze, heatmap affiancate degli algoritmi sullo stesso scenario, con i goal e lo start sovrapposti. Rende immediatamente visibile *dove* ciascuna strategia spreca visite (dead-end esplorati, corridoi ripercorsi) e come cambia il comportamento al crescere di `k`.

*Cosa faremo qui:* griglia di heatmap (righe = maze, colonne = algoritmo) per un valore fisso di `k`, più un confronto al variare di `k` sul maze più interessante.

## 6. Replanning e costo computazionale

Analisi dei campi `replanning_events`, `cumulative_planning_time_s` e `cumulative_nodes_expanded`: quante volte ogni algoritmo rivede il piano, quanto costa in tempo di calcolo e quanti nodi espande complessivamente. È il test diretto dell'ipotesi centrale della proposta: A* ri-pianifica da zero a ogni muro scoperto, mentre D*-Lite ripara il piano in modo incrementale.

*Cosa faremo qui:* numero di replanning per run, distribuzione dei tempi di pianificazione, confronto nodi espansi A* vs D*-Lite sullo stesso scenario.

## 7. Difficoltà dello scenario e robustezza

Usiamo il detour index dei goal come misura continua di difficoltà e osserviamo come degradano le metriche al suo crescere (e al crescere di `k`). La robustezza, come da proposta, è valutata sia come tasso di successo (l'agente raggiunge sempre il goal?) sia come pendenza della degradazione: un algoritmo robusto mantiene il 100% di successo e peggiora in modo contenuto sugli scenari difficili.

*Cosa faremo qui:* scatter/line plot di `total_moves` (e delle altre metriche chiave) in funzione del detour massimo dello scenario, per algoritmo; tabella del tasso di successo per livello di difficoltà.

## 8. Conclusioni

Sintesi dei risultati rispetto alle ipotesi della sezione 4 della proposta: quali sono state confermate, quali smentite e su quali maze/scenari le differenze tra strategie sono più marcate. Da qui estraiamo le figure e le tabelle da riportare nella relazione finale (`docs/Rob-26-MazeSolver_report.md`).